In [1]:
import pandas as pd

In [2]:
df_litho = pd.read_csv("C:\\Users\\vanmu\\copperbelt-mineral-prospectivity\\data\\grid_features_litho_contact_v02.csv")
df_litho.head()

,id,litho_contact_litho_class,distance,distance_2
0,4825,4,0.000000,0
1,4825,4,0.000000,0
2,4825,4,46.770524,0
3,4825,14,0.000000,0
4,4825,4,0.000000,0


In [5]:
df_litho.rename(columns={"Zonal Statistics_litho_majority": "litho_mojority",	"Joined layer_dist_litho_contact": "distance_litho_contact"}, inplace = True)

In [3]:
print(f"Original Rows (with QGIS overlap): {len(df_litho)}")
print(f"Unique Grid cells (Actual map size): {df_litho['id'].nunique()}")

Original Rows (with QGIS overlap): 23606
Unique Grid cells (Actual map size): 1899


In [11]:
df_base = pd.read_csv("C:\\Users\\vanmu\\copperbelt-mineral-prospectivity\\data\\copperbelt_base_grid.csv")

df_base.head()

,id,centroid_x,centroid_y,deposit_present
0,4825.0,529911.713578,8.502460e+06,0
1,4826.0,529382.796539,8.499679e+06,0
2,4827.0,529694.510138,8.496493e+06,0
3,4922.0,533039.846413,8.502730e+06,0
4,4923.0,532640.011800,8.499800e+06,0


In [16]:
# Display duplicate deposit rows in `df_base`
dep = df_base[df_base['deposit_present'] == 1]
dups = dep[dep.duplicated('id', keep=False)].sort_values('id')

print(f"Total deposit rows: {len(dep)}")
print(f"Unique deposit ids: {dep['id'].nunique()}")
print(f"Duplicate deposit rows: {len(dep) - dep['id'].nunique()}\n")

# show duplicated rows (all columns)
try:
    from IPython.display import display
    display(dups)
except Exception:
    print(dups.to_string(index=False))

# compact summary of duplicated ids and counts
dup_counts = dep.groupby('id').size().loc[lambda s: s > 1].sort_values(ascending=False)
print('\nDuplicated id counts:')
print(dup_counts.to_string())

Total deposit rows: 210
Unique deposit ids: 139
Duplicate deposit rows: 71



,id,centroid_x,centroid_y,deposit_present
2087,883.0,327640.011800,8.814800e+06,1
2086,883.0,327640.011800,8.814800e+06,1
2085,883.0,327640.011800,8.814800e+06,1
2091,979.0,332701.899062,8.818560e+06,1
2092,979.0,332701.899062,8.818560e+06,1
...,...,...,...,...
733,6852.0,632624.075686,8.554886e+06,1
818,7046.0,642489.923713,8.554860e+06,1
817,7046.0,642489.923713,8.554860e+06,1
986,7722.0,678217.522150,8.570018e+06,1



Duplicated id counts:
id
2432.0    10
2920.0     4
6158.0     4
6066.0     4
883.0      3
2540.0     3
2627.0     3
980.0      3
1948.0     3
5259.0     3
4881.0     3
4782.0     3
1949.0     2
2335.0     2
1951.0     2
1173.0     2
979.0      2
2726.0     2
2637.0     2
2630.0     2
2529.0     2
1560.0     2
3117.0     2
3408.0     2
3017.0     2
4675.0     2
4578.0     2
3697.0     2
4783.0     2
5354.0     2
5462.0     2
3312.0     2
3315.0     2
6064.0     2
5968.0     2
6067.0     2
6161.0     2
6248.0     2
6261.0     2
6358.0     2
6359.0     2
6455.0     2
6556.0     2
6558.0     2
6651.0     2
6750.0     2
6852.0     2
7046.0     2
7722.0     2


In [15]:
df_base['deposit_present'].sum()

np.int64(210)

In [14]:
# Deduplicate on id before merging so df_merged contains one row per unique grid cell id.
# df_litho and df_base both contain duplicate id rows, so we keep one representative row per id.
df_litho_unique = df_litho.drop_duplicates(subset='id', keep='first')
df_base_unique = df_base.drop_duplicates(subset='id', keep='first')[['id', 'deposit_present']]

# Merge unique ids from the lithology/contact dataset with deposit info from the base grid.
df_merged = df_litho_unique.merge(
    df_base_unique,
    on='id',
    how='left'
)

df_merged['has_deposit'] = df_merged['deposit_present'] == 1

print(f"Original lithology rows: {len(df_litho)}")
print(f"Unique lithology ids: {len(df_litho_unique)}")
print(f"Original base rows: {len(df_base)}")
print(f"Unique base ids: {len(df_base_unique)}")
print(f"Merged unique ids: {len(df_merged)}")
print(f"IDs with deposit info from base: {df_merged['deposit_present'].notna().sum()}")
print(f"Unique ids flagged as deposit present: {df_merged['has_deposit'].sum()}")

# Optionally, keep only rows that correspond to deposits if desired.
df_merged_deposits = df_merged[df_merged['has_deposit']].copy()
print(f"Unique ids with deposit present: {len(df_merged_deposits)}")

df_merged.head()

Original lithology rows: 23606
Unique lithology ids: 1899
Original base rows: 2108
Unique base ids: 1900
Merged unique ids: 1899
IDs with deposit info from base: 1899
Unique ids flagged as deposit present: 139
Unique ids with deposit present: 139


,id,litho_contact_litho_class,distance,distance_2,deposit_present,has_deposit
0,4825,4,0.000000,0,0,False
1,4826,14,2628.229476,0,0,False
2,4827,14,2628.229476,0,0,False
3,4922,4,0.000000,0,0,False
4,4923,4,0.000000,0,0,False


In [13]:
len(df_merged)

26771

In [ ]:
# Simple: drop unwanted columns and save the merged dataframe.
out_path = r"C:\Users\vanmu\copperbelt-mineral-prospectivity\data\copperbelt_merged_grid.csv"
df_to_save = df_merged.drop(columns=['distance_2', 'has_deposit'], errors='ignore')
df_to_save.to_csv(out_path, index=False)
print(f"Saved {len(df_to_save)} rows to {out_path}")

Saved 1899 rows to C:\Users\vanmu\copperbelt-mineral-prospectivity\data\copperbelt_merged_grid.csv


: 

## GRAVITY ANOMALIES

In [24]:
gravity = pd.read_csv(
    r"H:/My Drive/Research/GeoMining/Data/grid_-17.558_51.465_-34.85_37.35/grid_isostatic_-17.558_51.465_-34.85_37.35.txt",
    comment="#",
    sep=r"\s+",
    header=0,
    engine="python",
)
# normalize column names
gravity.columns = ["lon", "lat", "isostatic"]

print(gravity.head())
print(gravity.shape)

         lon        lat  isostatic
0 -17.566667  37.333333       6.50
1 -17.533333  37.333333       6.87
2 -17.500000  37.333333      11.19
3 -17.466667  37.333333      14.28
4 -17.433333  37.333333      20.69
(4490024, 3)


In [26]:
margin = 50000   # 50 km

xmin = grid.centroid_x.min() - margin
xmax = grid.centroid_x.max() + margin

ymin = grid.centroid_y.min() - margin
ymax = grid.centroid_y.max() + margin

gravity = gravity[
    (gravity.x >= xmin) &
    (gravity.x <= xmax) &
    (gravity.y >= ymin) &
    (gravity.y <= ymax)
]

print(gravity.shape)

(23167, 5)


In [28]:
grid.to_csv(
    "C:/Users/vanmu/copperbelt-mineral-prospectivity/data/copperbelt_with_isostic.csv",
    index=False
)

In [32]:
cmdb = pd.read_csv(
    r"H:\My Drive\Research\GeoMining\Data\CMDB_Data.csv",
    encoding="cp1252"
)
print(cmdb.shape)
cmdb.head()
print(cmdb.shape)
cmdb.head()

(1366, 103)
(1366, 103)


,LAB_ID,PREVIOUS_LAB_ID1,PREVIOUS_LAB_ID2,PREVIOUS_LAB_ID3,FIELD_ID,JOB_ID,PREVIOUS_JOB_ID1,PREVIOUS_JOB_ID2,PREVIOUS_JOB_ID3,SUBMITTER,...,Th_ppm_MS_ST,Tl_ppm_MS_ST,Tm_ppm_MS_ST,U_ppm_MS_ST,V_ppm_AES_ST,W_ppm_MS_ST,Y_ppm_MS_ST,Yb_ppm_MS_ST,Zn_ppm_AES_ST,Zr_ppm_AES_ST
0,C355417,NaN,NaN,NaN,RM0001,MRP11968,NaN,NaN,NaN,Rare Metals Task,...,0.2,-0.5,-0.05,0.30,51.0,-1.0,-0.5,-0.1,1290.0,3.8
1,C360759,NaN,NaN,NaN,RM0027,MRP12307,NaN,NaN,NaN,Rare Metals Task,...,9.7,0.5,-0.05,1.75,24.0,28.0,2.3,0.3,-5.0,133.0
2,C360762,NaN,NaN,NaN,RM0030,MRP12307,NaN,NaN,NaN,Rare Metals Task,...,2.6,-0.5,0.08,0.63,-5.0,22.0,5.9,0.6,161.0,16.2
3,C360763,NaN,NaN,NaN,RM0031,MRP12307,NaN,NaN,NaN,Rare Metals Task,...,0.2,-0.5,-0.05,34.80,493.0,11.0,1.9,0.2,29.0,19.1
4,C360769,NaN,NaN,NaN,RM0037,MRP12307,NaN,NaN,NaN,Rare Metals Task,...,2.6,-0.5,0.22,31.20,68.0,8.0,13.0,1.4,4480.0,150.0


In [33]:
cmdb["COUNTRY"].value_counts()

COUNTRY
United States                   1021
Canada                            59
Chile                             53
Australia                         39
Sweden                            38
Mexico                            30
Peru                              19
Japan                             18
Brazil                            17
South Africa                      10
Zambia                             7
Vietnam                            6
Indonesia                          4
Finland                            4
Poland                             4
Democratic Republic of Congo       4
Mauritania                         4
Portugal                           4
Argentina                          4
Namibia                            3
China                              3
Philippines                        3
Norway                             2
Cuba                               2
Germany                            2
Russia                             1
Burma                         

In [1]:
import rasterio
import pandas as pd
from pyproj import Transformer

In [4]:
grid = pd.read_csv("C:\\Users\\vanmu\\copperbelt-mineral-prospectivity\\data\\copperbelt_base_grid.csv")

In [6]:
src = rasterio.open("H:/My Drive/Research/GeoMining/Data/EMAG2_V3_UpCont_DataTiff.tif")
print(src)
print(src.bounds)

<open DatasetReader name='H:/My Drive/Research/GeoMining/Data/EMAG2_V3_UpCont_DataTiff.tif' mode='r'>
BoundingBox(left=-0.0166666666675, bottom=-89.9833333378325, right=359.9833333513325, top=89.9833333378325)


In [13]:
transformer = Transformer.from_crs(
    "EPSG:32735",
    "EPSG:4326",
    always_xy= True
)

lon, lat = transformer.transform(
    grid.centroid_x.values,
    grid.centroid_y.values
)

In [14]:
coords = list(zip(lon, lat))

emag = [
    value[0]
    for value in src.sample(coords)
]

grid["emag"] = emag

In [15]:
grid.to_csv(
    "C:\\Users\\vanmu\\copperbelt-mineral-prospectivity\\data\\copperbelt_with_emag.csv",
    index=False
)